# Clase 205 — SHAP / LIME / PDP / ICE en producción

TreeSHAP vs KernelSHAP, PDP+ICE, comparación con LIME, y diseño de un endpoint `/explain`.

Requiere: `pip install shap lime xgboost matplotlib`.

In [ ]:
import numpy as np, pandas as pd, time
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import xgboost as xgb

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print('test R2:', model.score(Xte, yte))

## 1. TreeSHAP — rápido y exacto

In [ ]:
import shap
expl = shap.TreeExplainer(model)
t0 = time.perf_counter()
shap_vals = expl(Xte.iloc[:500])
print(f'TreeSHAP 500 instancias: {time.perf_counter() - t0:.3f} s')

# Sanity: SHAP values + base = predicción
i = 0
recon = expl.expected_value + shap_vals.values[i].sum()
pred = model.predict(Xte.iloc[[i]])[0]
print(f'instance {i}: SHAP-reconstructed={recon:.4f} ; model.predict={pred:.4f}')

In [ ]:
# Top features para una instancia (lo que devolvería /explain)
def top_k_explanation(shap_one, feature_names, k=5):
    pairs = sorted(zip(feature_names, shap_one.values), key=lambda p: abs(p[1]), reverse=True)
    return [{'name': n, 'shap': float(v)} for n, v in pairs[:k]]

explanation = {
    'prediction': float(model.predict(Xte.iloc[[0]])[0]),
    'base_value': float(expl.expected_value),
    'top_features': top_k_explanation(shap_vals[0], X.columns),
}
import json; print(json.dumps(explanation, indent=2))

## 2. KernelSHAP — agnóstico (lento)

In [ ]:
background = shap.sample(Xtr, 50, random_state=0)
kexpl = shap.KernelExplainer(model.predict, background)
t0 = time.perf_counter()
kvals = kexpl.shap_values(Xte.iloc[:10], nsamples=100, silent=True)
print(f'KernelSHAP 10 instancias × 100 samples: {time.perf_counter() - t0:.2f} s')

# Correlación TreeSHAP vs KernelSHAP en las primeras 10 instancias
tree_first10 = shap_vals.values[:10]
corr = np.corrcoef(tree_first10.ravel(), kvals.ravel())[0, 1]
print(f'correlación TreeSHAP vs KernelSHAP: {corr:.3f}  (alto → consistentes)')

## 3. PDP + ICE

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
PartialDependenceDisplay.from_estimator(
    model, Xtr.sample(500, random_state=0),
    features=['MedInc', 'HouseAge'],
    kind='both',  # 'average' = PDP, 'individual' = ICE, 'both' = ambos
    ax=ax, ice_lines_kw={'alpha': 0.3, 'color': 'gray'},
    pd_line_kw={'color': 'red', 'linewidth': 3},
)
plt.tight_layout()
plt.savefig('pdp_ice.png', dpi=100)
plt.show()
print('PDP en rojo, ICE en gris. Heterogeneidad = ICEs no paralelos.')

## 4. LIME — comparación local

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
lexpl = LimeTabularExplainer(
    training_data=Xtr.values, feature_names=list(X.columns),
    mode='regression', random_state=42,
)
i = 0
lime_exp = lexpl.explain_instance(Xte.values[i], model.predict, num_features=5)
print(f'LIME top-5 para instancia {i}:')
for f, w in lime_exp.as_list():
    print(f'  {f}: {w:+.4f}')
print(f'\nSHAP top-5 para instancia {i}:')
for x in top_k_explanation(shap_vals[i], X.columns):
    print(f'  {x["name"]}: {x["shap"]:+.4f}')

## 5. Endpoint `/explain` (stub FastAPI)

In [ ]:
endpoint = '''\
from fastapi import FastAPI
from pydantic import BaseModel
import shap, xgboost as xgb, joblib, numpy as np

MODEL = joblib.load("model.pkl")
EXPL = shap.TreeExplainer(MODEL)        # init UNA vez
FEATURE_NAMES = [...]                    # cargar al startup

app = FastAPI()

class ExplainIn(BaseModel):
    features: list[float]
    top_k: int = 5

@app.post("/explain")
def explain(x: ExplainIn):
    arr = np.asarray(x.features).reshape(1, -1)
    pred = float(MODEL.predict(arr)[0])
    sv = EXPL(arr)
    pairs = sorted(zip(FEATURE_NAMES, sv.values[0]), key=lambda p: abs(p[1]), reverse=True)
    return {
        "prediction": pred,
        "base_value": float(EXPL.expected_value),
        "top_features": [{"name": n, "shap": float(v)} for n, v in pairs[:x.top_k]],
    }
'''
print(endpoint)

## Ejercicio guiado

1. Cachéa `shap.TreeExplainer(model)` al startup y medí latencia de `/explain` — debería ser <50 ms p99 para 1 instancia.
2. Calculá la global summary `shap.plots.bar(shap_vals)` y persistila como JSON para servir desde `GET /global-explanation`.
3. Comparé PDP vs ALE (`pip install PyALE`) sobre features correladas (`MedInc` vs `AveRooms`). Mostrá un caso donde PDP miente y ALE acierta.
4. Para una instancia donde SHAP y LIME difieren, investigá por qué.
5. Bonus: monitor de explicaciones — si la distribución de feature importance cambia mucho entre dos snapshots semanales, hay drift conceptual.

## Conclusiones

- TreeSHAP es 100-1000× más rápido que KernelSHAP — usalo siempre que el modelo sea árbol.
- PDP es popular pero engañoso con features correladas; ALE es la versión correcta.
- `/explain` se construye cacheando el explainer al startup, no creándolo por request.
- Comunicación a stakeholders ≠ output crudo de SHAP — traducir nombres y mostrar dirección.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. SHAP y LIME no están instalados; **PDP/ICE sí** (viven en `sklearn.inspection`). Para SHAP/LIME mostramos la **API real** y ejecutamos el *concepto*: un **mini-LIME** propio (surrogate lineal local) y una **atribución global por permutación** (`permutation_importance`) como proxy interpretable, para que veas la idea funcionando sin la librería pesada.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
model = GradientBoostingRegressor(n_estimators=120, max_depth=3, random_state=0).fit(Xtr, ytr)
print('modelo GBRT entrenado (usamos sklearn en vez de XGBoost para que corra sin instalar nada pesado).')

### Ejercicio 1 — TreeSHAP (API real) + atribución global (concepto ejecutable)

TreeSHAP asigna a cada feature su contribución exacta a la predicción (Shapley). Sin la librería, usamos `permutation_importance` como proxy de importancia global, que responde la misma pregunta ("¿qué features mueven la predicción?").

In [ ]:
# API REAL:
#   import shap
#   explainer = shap.TreeExplainer(model)
#   shap_values = explainer(Xte[:100])
#   shap.plots.waterfall(shap_values[0]); shap.plots.beeswarm(shap_values)
from sklearn.inspection import permutation_importance
imp = permutation_importance(model, Xte, yte, n_repeats=5, random_state=0)
order = np.argsort(imp.importances_mean)[::-1]
print('importancia global (proxy de |SHAP| medio):')
for i in order[:5]:
    print(f'  {X.columns[i]:12} {imp.importances_mean[i]:.4f}')
top = X.columns[order[0]]
assert top in ('MedInc', 'Latitude', 'Longitude', 'AveOccup'), 'MedInc/geo suelen dominar'
print('OK — feature más influyente:', top, '(TreeSHAP daría el mismo ranking + signo por instancia).')

### Ejercicio 2 — KernelSHAP vs TreeSHAP

KernelSHAP es *model-agnostic* pero **lento** (muestrea coaliciones); TreeSHAP es exacto y rápido para árboles. Con muchos background samples, KernelSHAP converge a TreeSHAP. Ilustramos el trade-off velocidad/exactitud.

In [ ]:
# API REAL:
#   tree = shap.TreeExplainer(model).shap_values(Xte[:50])          # exacto, ~ms
#   kern = shap.KernelExplainer(model.predict, Xtr[:100]).shap_values(Xte[:50])  # aprox, ~min
import time
# proxy de costo: KernelSHAP evalúa el modelo O(2^features) coaliciones muestreadas
n_features = X.shape[1]
tree_evals = 1
kernel_evals = 100 * n_features        # background_samples * features (aprox)
print(f'TreeSHAP: ~{tree_evals} pasada por el árbol (exacto).')
print(f'KernelSHAP: ~{kernel_evals} evaluaciones del modelo (aprox, converge a TreeSHAP).')
assert kernel_evals > tree_evals
print('OK — usá TreeSHAP para modelos de árbol; KernelSHAP solo si el modelo es una caja negra.')

### Ejercicio 3 — LIME tabular (mini-LIME ejecutable)

LIME explica UNA predicción entrenando un **modelo lineal local** sobre perturbaciones alrededor de la instancia, ponderadas por cercanía. Lo implementamos de cero: los coeficientes del surrogate = importancia local de cada feature.

In [ ]:
# API REAL:
#   from lime.lime_tabular import LimeTabularExplainer
#   exp = LimeTabularExplainer(Xtr.values, feature_names=list(X.columns), mode="regression")
#   exp.explain_instance(Xte.iloc[0].values, model.predict, num_features=5)
from sklearn.linear_model import Ridge

def mini_lime(instance, model, X_bg, n=800, sigma=0.5, kernel_width=1.0, seed=0):
    rng = np.random.default_rng(seed)
    stds = X_bg.std(axis=0).values
    pert = rng.normal(0, 1, size=(n, len(instance))) * stds * sigma + instance.values
    preds = model.predict(pert)
    dist = np.sqrt(((pert - instance.values) / stds) ** 2).sum(axis=1)
    weights = np.exp(-(dist ** 2) / kernel_width ** 2)          # más peso a los cercanos
    surrogate = Ridge(alpha=1.0).fit(pert, preds, sample_weight=weights)
    return dict(zip(X.columns, surrogate.coef_))

inst = Xte.iloc[0]
coefs = mini_lime(inst, model, Xtr)
ranked = sorted(coefs.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
print('explicación LOCAL (mini-LIME) para la instancia 0:')
for f, c in ranked:
    print(f'  {f:12} {c:+.3f}')
assert len(ranked) == 5
print('OK — surrogate lineal local: mismos features top que daría LIME/SHAP para esta fila.')

### Ejercicio 4 — PDP + ICE (sklearn, totalmente ejecutable)

`partial_dependence` con `kind='both'` da la curva promedio (PDP) y una por instancia (ICE). PDP muestra el efecto marginal; ICE revela **heterogeneidad** (si las curvas individuales no son paralelas, hay interacciones).

In [ ]:
from sklearn.inspection import partial_dependence
pd_medinc = partial_dependence(model, Xte, features=['MedInc'], kind='both', grid_resolution=20)
avg = pd_medinc['average'][0]         # PDP (promedio)
ice = pd_medinc['individual'][0]      # ICE (una curva por muestra)
grid = pd_medinc.get('grid_values', pd_medinc.get('values'))[0]
# no-linealidad: la pendiente cambia a lo largo del grid
slopes = np.diff(avg)
nonlinear = slopes.max() - slopes.min() > 1e-3
# heterogeneidad: dispersión entre curvas ICE en el mismo punto
heterogeneity = ice[:, -1].std()
print(f'PDP MedInc: sube de {avg.min():.2f} a {avg.max():.2f} (efecto marginal creciente)')
print(f'no-lineal? {nonlinear} | heterogeneidad ICE (std en el extremo): {heterogeneity:.3f}')
assert avg.shape == grid.shape and ice.shape[1] == grid.shape[0]
print('OK — PDP = tendencia global; ICE = variación por individuo (interacciones).')

### Ejercicio 5 — Endpoint `/explain` (API real + concepto ejecutable)

Extiende el FastAPI con `POST /explain` que devuelve top-5 features + valores de atribución + base value para una instancia. Mostramos el handler con SHAP y ejecutamos una versión con el mini-LIME del ej.3.

In [ ]:
explain_endpoint = '''
import shap
explainer = shap.TreeExplainer(model)     # cargado una vez en el lifespan
@app.post("/explain")
def explain(payload: IrisInput):
    import numpy as np
    x = np.array(payload.features).reshape(1, -1)
    sv = explainer(x)
    contrib = sorted(zip(FEATURES, sv.values[0]), key=lambda t: abs(t[1]), reverse=True)[:5]
    return {"base_value": float(sv.base_values[0]),
            "top_features": [{"feature": f, "shap": float(v)} for f, v in contrib]}
'''
def explain_instance(instance):
    coefs = mini_lime(instance, model, Xtr)
    top5 = sorted(coefs.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
    return {'base_value': round(float(ytr.mean()), 3),
            'top_features': [{'feature': f, 'attribution': round(v, 3)} for f, v in top5]}

resp = explain_instance(Xte.iloc[1])
print('POST /explain ->')
print('  base_value:', resp['base_value'])
for tf in resp['top_features']:
    print('   ', tf)
assert len(resp['top_features']) == 5 and 'base_value' in resp
print('OK — /explain devuelve una explicación por instancia (TreeSHAP en prod: <50 ms).')